In [0]:
# Databricks notebook source
%pip install streamlit

In [0]:
# Restart Python environment để nhận thư viện mới
dbutils.library.restartPython()

In [0]:
# 🚲 Customer Churn Prediction & Retention App
# Note: To run as a Streamlit app outside notebooks, use: streamlit run streamlit_app.py
# This version works in Databricks notebooks using widgets

import pandas as pd

# Load predictions from Gold table
df = spark.table("workspace.gold.customer_churn_predictions").toPandas()

# Create widget for country selection
dbutils.widgets.dropdown("country", df["country"].iloc[0], [str(x) for x in df["country"].unique()])
country = dbutils.widgets.get("country")

# Filter by selected country
filtered_df = df[df["country"] == country]

# Display Summary
print(f"📊 Customer Churn Analysis for Country: {country}")
print(f"\n🎯 Total High Risk Customers: {len(filtered_df[filtered_df['predicted_churn_risk'] == 1])}")
print(f"\n📋 Filtered Results ({len(filtered_df)} customers):")

# Display the data
display(filtered_df[["customer_key", "monetary", "recency", "churn_probability"]])

In [0]:
# 📊 Customer Churn Analysis - Detailed View

# Calculate summary metrics
total_high_risk = len(filtered_df[filtered_df["predicted_churn_risk"] == 1])
at_risk_revenue = filtered_df[filtered_df["predicted_churn_risk"] == 1]["monetary"].sum()

# Display summary
print("📊 Customer Churn Analysis Summary")
print(f"\n🎯 Total High Risk Customers: {total_high_risk:,}")
print(f"💰 Total Revenue at Risk: ${at_risk_revenue:,.0f}")
print("\n" + "="*50)
print("📋 Top High-Value At-Risk Customers (Immediate Action Required):")
print("="*50 + "\n")

# Filter Top 10 high-risk customers with highest spend
top_at_risk = filtered_df[filtered_df["predicted_churn_risk"] == 1].sort_values(
    by=["monetary", "churn_probability"], ascending=[False, False]
).head(10)

# Display the table
display(top_at_risk[["customer_key", "monetary", "frequency", "recency", "churn_probability"]])